
# 03 · Dynamic Pricing Avanzado — Modelo ML básico (ADR)
**Objetivo:** Entrenar un modelo supervisado para **predecir ADR** por reserva/fecha usando variables de calendario, demanda y segmentos.  
**Entradas:**  
- `data/processed/hotel_bookings_clean_for_bi.csv` (del notebook 01)  
- `holidays_spain_2025.csv` (festivos)  
- `local_events_template.csv` (eventos locales con `weight`)  
- *(opcional)* `date_table_daily_2020_2027.csv` para generar calendario futuro

**Salidas:**  
- Modelo: `models/adr_model.pkl`  
- Métricas del test temporal (MAE, MAPE, R²)  
- Predicciones test: `data/processed/adr_predictions_test.csv`  
- Predicciones a futuro (N días): `data/processed/adr_predictions_future.csv`


In [ ]:


import os ,warnings ,numpy as np ,pandas as pd 
from datetime import datetime ,timedelta 

from sklearn .model_selection import train_test_split 
from sklearn .preprocessing import OneHotEncoder 
from sklearn .compose import ColumnTransformer 
from sklearn .pipeline import Pipeline 
from sklearn .metrics import mean_absolute_error ,r2_score 


try :
    from lightgbm import LGBMRegressor 
    _HAS_LGBM =True 
except Exception :
    _HAS_LGBM =False 

try :
    from xgboost import XGBRegressor 
    _HAS_XGB =True 
except Exception :
    _HAS_XGB =False 

from sklearn .ensemble import RandomForestRegressor 
import joblib 
import matplotlib .pyplot as plt 

warnings .filterwarnings ("ignore")


CWD =os .getcwd ()
PROJ =os .path .abspath (os .path .join (CWD ,".."))if os .path .basename (CWD ).lower ()=="notebooks"else CWD 
DATA_PROC =os .path .join (PROJ ,"data","processed")
MODELS_DIR =os .path .join (PROJ ,"models")
os .makedirs (MODELS_DIR ,exist_ok =True )


HOTEL_BI =os .path .join (DATA_PROC ,"hotel_bookings_clean_for_bi.csv")
HOLIDAYS =os .path .join (PROJ ,"holidays_spain_2025.csv")
EVENTS =os .path .join (PROJ ,"local_events_template.csv")
DATE_TABLE =os .path .join (PROJ ,"date_table_daily_2020_2027.csv")

HOTEL_BI ,HOLIDAYS ,EVENTS ,DATE_TABLE 


In [ ]:


assert os .path .exists (HOTEL_BI ),"Falta data/processed/hotel_bookings_clean_for_bi.csv"
df =pd .read_csv (HOTEL_BI ,parse_dates =["arrival_date"])

hol =pd .read_csv (HOLIDAYS ,parse_dates =["date"])if os .path .exists (HOLIDAYS )else pd .DataFrame (columns =["date","name","scope"])
evt =pd .read_csv (EVENTS ,parse_dates =["date"])if os .path .exists (EVENTS )else pd .DataFrame (columns =["date","city","event_name","weight"])

df .head (3 ),hol .head (3 ),evt .head (3 )



## 3) Preparación de variables (features)
- Target: `adr` **solo en reservas NO canceladas** (`is_canceled==0`).  
- Features: `lead_time`, `stay_nights`, `guests`, calendario (`dow`, `month`, `is_weekend`), segmentos (`hotel`, `market_segment`, `country`, `reserved_room_type`, `assigned_room_type`), festivos y eventos.


In [ ]:


work =df [df ["is_canceled"]==0 ].copy ()
work =work [work ["adr"].notna ()&(work ["adr"]>=0 )]


work ["dow"]=work ["arrival_date"].dt .dayofweek 
work ["month"]=work ["arrival_date"].dt .month 
work ["is_weekend"]=work ["dow"].isin ([4 ,5 ,6 ]).astype (int )


hol_days =set (hol ["date"].dt .normalize ().tolist ())if len (hol )else set ()
work ["is_holiday"]=work ["arrival_date"].isin (hol_days ).astype (int )

evt_daily =evt .groupby ("date",as_index =False )["weight"].sum ()if len (evt )else pd .DataFrame ({"date":[],"weight":[]})
evt_daily ["date"]=pd .to_datetime (evt_daily ["date"]).dt .normalize ()if len (evt_daily )else evt_daily 
work =work .merge (evt_daily .rename (columns ={"date":"arrival_date","weight":"event_weight"}),
on ="arrival_date",how ="left")
work ["event_weight"]=work ["event_weight"].fillna (0.0 )


target_col ="adr"
num_cols =["lead_time","stay_nights","guests","dow","month","is_weekend","is_holiday","event_weight"]
cat_cols =["hotel","market_segment","country","reserved_room_type","assigned_room_type"]

X =work [num_cols +cat_cols ].copy ()
y =work [target_col ].astype (float ).values 

X .head (3 ),y [:3 ]



## 4) Partición temporal (train/test)
Usamos una **partición temporal simple**: último 20% del rango de fechas como **test**.


In [ ]:


cut_date =work ["arrival_date"].quantile (0.8 )
train_idx =work ["arrival_date"]<cut_date 
test_idx =work ["arrival_date"]>=cut_date 

X_train ,y_train =X [train_idx ],y [train_idx ]
X_test ,y_test =X [test_idx ],y [test_idx ]


pre =ColumnTransformer (
transformers =[
("num","passthrough",num_cols ),
("cat",OneHotEncoder (handle_unknown ="ignore",sparse_output =False ),cat_cols )
]
)


if _HAS_LGBM :
    model =LGBMRegressor (n_estimators =600 ,learning_rate =0.05 ,num_leaves =31 ,subsample =0.9 ,colsample_bytree =0.9 ,random_state =42 )
elif _HAS_XGB :
    model =XGBRegressor (n_estimators =800 ,max_depth =6 ,learning_rate =0.05 ,subsample =0.9 ,colsample_bytree =0.9 ,random_state =42 ,tree_method ="hist")
else :
    model =RandomForestRegressor (n_estimators =400 ,random_state =42 ,n_jobs =-1 )

pipe =Pipeline ([("prep",pre ),("model",model )])

pipe .fit (X_train ,y_train )

pred_test =pipe .predict (X_test )
mae =mean_absolute_error (y_test ,pred_test )
mape =np .mean (np .abs ((y_test -pred_test )/np .clip (y_test ,1e-6 ,None )))*100 
r2 =r2_score (y_test ,pred_test )

mae ,mape ,r2 



## 5) Evaluación visual (opcional)


In [ ]:

plt .figure (figsize =(6 ,6 ))
plt .scatter (y_test ,pred_test ,alpha =0.3 )
plt .xlabel ("ADR real (test)")
plt .ylabel ("ADR predicho (test)")
plt .title ("Predicho vs Real — Test")
lims =[min (y_test .min (),pred_test .min ()),max (y_test .max (),pred_test .max ())]
plt .plot (lims ,lims )
plt .show ()



## 6) Guardar modelo y predicciones


In [ ]:


model_path =os .path .join (MODELS_DIR ,"adr_model.pkl")
import joblib ;joblib .dump (pipe ,model_path )


out_test =work .loc [test_idx ,["arrival_date"]+num_cols +cat_cols ].copy ()
out_test ["adr_real"]=y_test 
out_test ["adr_pred"]=pred_test 
out_test_path =os .path .join (DATA_PROC ,"adr_predictions_test.csv")
out_test .to_csv (out_test_path ,index =False )

model_path ,out_test_path 



## 7) Predicción a futuro (próximos N días)
Si tienes `date_table_daily_2020_2027.csv`, generamos un calendario para **los próximos 60 días** y predecimos ADR sugerido según el modelo.


In [ ]:

FUTURE_DAYS =60 

if os .path .exists (DATE_TABLE ):
    date_tbl =pd .read_csv (DATE_TABLE ,parse_dates =["Date","MonthStart"])
    today =pd .Timestamp .today ().normalize ()
    future =date_tbl [(date_tbl ["Date"]>=today )&(date_tbl ["Date"]<today +pd .Timedelta (days =FUTURE_DAYS ))].copy ()
else :
    today =pd .Timestamp .today ().normalize ()
    future =pd .DataFrame ({"Date":pd .date_range (today ,periods =FUTURE_DAYS ,freq ="D")})
    future ["MonthStart"]=future ["Date"].values .astype ("datetime64[M]")
    future ["Year"]=future ["Date"].dt .year 
    future ["MonthNumber"]=future ["Date"].dt .month 
    future ["WeekdayNumber"]=future ["Date"].dt .weekday 
    future ["WeekdayName"]=future ["Date"].dt .day_name ()
    future ["IsWeekend"]=future ["WeekdayNumber"].isin ([4 ,5 ,6 ])


most_hotel =work ["hotel"].mode ()[0 ]if "hotel"in work and len (work ["hotel"].dropna ())else "Unknown"
most_market =work ["market_segment"].mode ()[0 ]if "market_segment"in work and len (work ["market_segment"].dropna ())else "Unknown"
most_country =work ["country"].mode ()[0 ]if "country"in work and len (work ["country"].dropna ())else "Unknown"
most_rrt =work ["reserved_room_type"].mode ()[0 ]if "reserved_room_type"in work and len (work ["reserved_room_type"].dropna ())else "U"
most_art =work ["assigned_room_type"].mode ()[0 ]if "assigned_room_type"in work and len (work ["assigned_room_type"].dropna ())else "U"

future_df =pd .DataFrame ({
"arrival_date":future ["Date"],
"lead_time":30 ,
"stay_nights":2 ,
"guests":2 ,
"dow":future ["WeekdayNumber"],
"month":future ["MonthNumber"],
"is_weekend":future ["IsWeekend"].astype (int )
})


if len (hol ):
    hol_days =set (hol ["date"].dt .normalize ().tolist ())
    future_df ["is_holiday"]=future_df ["arrival_date"].isin (hol_days ).astype (int )
else :
    future_df ["is_holiday"]=0 

if len (evt ):
    evt_daily =evt .groupby ("date",as_index =False )["weight"].sum ()
    evt_daily ["date"]=pd .to_datetime (evt_daily ["date"]).dt .normalize ()
    future_df =future_df .merge (evt_daily .rename (columns ={"date":"arrival_date","weight":"event_weight"}),
    on ="arrival_date",how ="left")
    future_df ["event_weight"]=future_df ["event_weight"].fillna (0.0 )
else :
    future_df ["event_weight"]=0.0 


future_df ["hotel"]=most_hotel 
future_df ["market_segment"]=most_market 
future_df ["country"]=most_country 
future_df ["reserved_room_type"]=most_rrt 
future_df ["assigned_room_type"]=most_art 


future_df ["adr_pred"]=pipe .predict (future_df [["lead_time","stay_nights","guests","dow","month","is_weekend","is_holiday","event_weight",
"hotel","market_segment","country","reserved_room_type","assigned_room_type"]])

future_out =future_df [["arrival_date","adr_pred","is_weekend","is_holiday","event_weight","lead_time","stay_nights","guests","hotel","market_segment"]].copy ()
future_out_path =os .path .join (DATA_PROC ,"adr_predictions_future.csv")
future_out .to_csv (future_out_path ,index =False )

future_out_path ,future_out .head (3 )
